### PromptTemplate 
* [PromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.prompt.PromptTemplate.html#langchain_core.prompts.prompt.PromptTemplate)
* [ChatPromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html#langchain_core.prompts.chat.ChatPromptTemplate)
* [ChatMessagePromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatMessagePromptTemplate.html#langchain_core.prompts.chat.ChatMessagePromptTemplate)
* [FewShotPromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.few_shot.FewShotPromptTemplate.html#langchain_core.prompts.few_shot.FewShotPromptTemplate)
* PartialPrompt

In [ ]:
# uv add python-dotenv langchain langchain-openai

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


#### LLM 설정 (Groq / OpenAI 선택)

- **Groq**와 **OpenAI** 두 가지 모델을 모두 사용할 수 있습니다. (Groq는 OpenAI 호환 API라서 `ChatOpenAI`로 연결)
- 아래 셀의 `PROVIDER` 값(`"groq"` 또는 `"openai"`)만 바꾸면 이후 모든 예제의 모델이 바뀝니다. 값을 바꾼 뒤에는 이 셀부터 다시 실행하세요.
- 사용할 제공자의 키(`GROQ_API_KEY` 또는 `OPENAI_API_KEY`)가 `.env`에 있어야 합니다. OpenAI를 선택하면 사용 요금이 발생합니다.

In [18]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-5.6-luna",  # 대안: "gpt-4o" "gpt-4o-mini" 
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
    )

##### 1) PromptTemplate 의 from_template() 함수 사용
* 주로 LLM(텍스트 완성형 모델, ex. Ollama, GPT-3.5)과 함께 사용
* 하나의 문자열 프롬프트를 생성

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from pprint import pprint

template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

llm = get_llm(temperature=0.7)  # 제공자/키/모델명은 공통 설정 셀의 get_llm 사용
print("llm 설정 확인 완료", f"{PROVIDER} / {llm.model_name}")

llm 설정 확인 완료 groq / openai/gpt-oss-120b


In [7]:

chain = prompt_template | llm | StrOutputParser()
response = chain.invoke({"model_name":"ChatGPT", "count":3})
pprint(response)

('ChatGPT는 인터넷에 존재하는 방대한 텍스트 데이터를 수집해, 문맥에 맞는 다음 토큰을 예측하도록 하는 자기지도 '
 '학습(unsupervised pretraining)으로 기본 언어 모델을 학습합니다.  \n'
 '이후 인간이 작성한 질문‑답변 쌍을 이용해 지도학습(supervised fine‑tuning)으로 모델을 구체적인 대화 형식에 맞게 '
 '조정합니다.  \n'
 '마지막으로 인간 피드백을 보상 신호로 활용한 강화학습(RLHF)을 적용해, 보다 유익하고 안전한 응답을 생성하도록 최적화합니다.')


##### 2) PromptTemplate 결합하기
* 동일한 Prompt 패턴을 사용하지만 여러 개의 질문을 작성해서 LLM을 실행할 수도 있습니다.

In [8]:
template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

# 문자열 템플릿 결합 (PromptTemplate + PromptTemplate + 문자열)
combined_prompt = (
              prompt_template
              + PromptTemplate.from_template("\n\n 그리고 {model_name} 모델의 장점을 요약 정리해 주세요")
              + "\n\n {model_name} 모델과 비슷한 AI 모델은 어떤 것이 있나요? 모델명은 {language}로 답변해 주세요."
)
combined_prompt.format(model_name="ChatGPT", count=3, language="한국어")
print(combined_prompt)

chain = combined_prompt | llm | StrOutputParser()
response = chain.invoke({"model_name":"Gemini", "count":3, "language":"한국어"})

pprint(response)

input_variables=['count', 'language', 'model_name'] input_types={} partial_variables={} template='{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요.\n\n 그리고 {model_name} 모델의 장점을 요약 정리해 주세요\n\n {model_name} 모델과 비슷한 AI 모델은 어떤 것이 있나요? 모델명은 {language}로 답변해 주세요.'
('**Gemini 모델의 학습 원리 (3문장)**  \n'
 '1. Gemini는 대규모 텍스트와 멀티모달 데이터(이미지·영상·음성 등)를 사용해 사전 학습(pre‑training)하고, '
 'Transformer 기반의 다층 인코더·디코더 구조로 입력을 토큰화해 의미를 추출합니다.  \n'
 '2. 사전 학습 단계에서는 마스크드 언어 모델링, 다음 문장 예측, 이미지‑텍스트 정합성 학습 등 다양한 자기 '
 '지도(self‑supervised) 목표를 동시에 최적화해 다중 도메인 이해 능력을 기릅니다.  \n'
 '3. 이후 특정 작업(질문‑응답, 번역, 코딩 등)에 맞춰 소량의 라벨 데이터로 미세 조정(fine‑tuning)하거나, 프롬프트 '
 '엔지니어링을 활용해 제로·샷 학습을 수행합니다.  \n'
 '\n'
 '**Gemini 모델의 주요 장점**  \n'
 '- **멀티모달 통합**: 텍스트와 이미지·영상·음성 정보를 동시에 처리해 보다 풍부하고 일관된 응답을 생성합니다.  \n'
 '- **스케일 효율성**: 대규모 파라미터와 고효율 토큰화·샘플링 기법을 결합해 높은 정확도와 빠른 추론 속도를 동시에 달성합니다.  \n'
 '- **안전·윤리 강화**: 사전 학습 단계에서 편향·유해 콘텐츠를 억제하는 정교한 필터링과 인간 피드백(RLHF) 과정을 적용해 신뢰성 '
 '높은 결과를 제공합니다.  \n'
 '\n'
 '**Gemini와 비슷한 AI 모델 (한국어 명칭)**  \n'
 '- *

#### PromptTemplate 의 파라미터를 배열 형태로 하여 여러개 사용하는 경우

In [9]:
# List Comprehension 예제
# before (적용하지 않은 경우)
result_list = []
for val in range(10):
    # 2의
    if val % 2 == 0:
        result_list.append(val ** 2)

result_list        

[0, 4, 16, 36, 64]

In [10]:
# after (적용한 경우)
result_list2 = [val**2 for val in range(10) if val % 2 == 0]
result_list2

[0, 4, 16, 36, 64]

In [12]:
template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

questions = [
    {"model_name": "GPT-4", "count": 3}, #dict
    {"model_name": "Gemini", "count": 4},
    {"model_name": "Claude", "count": 5},
]

# 여러 개의 프롬프트를 미리 생성
# q변수가 dict 타입
formatted_prompts = [prompt_template.format(**q) for q in questions]
print(formatted_prompts)  # 미리 생성된 질문 목록 확인

for prompt in formatted_prompts:
    print(type(prompt), prompt)
    response = llm.invoke(prompt)
    pprint(response.content)
    print("-" * 50)

['GPT-4 모델의 학습 원리를 3 문장으로 한국어로 답변해 주세요.', 'Gemini 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.', 'Claude 모델의 학습 원리를 5 문장으로 한국어로 답변해 주세요.']
<class 'str'> GPT-4 모델의 학습 원리를 3 문장으로 한국어로 답변해 주세요.
('GPT‑4는 방대한 인터넷 텍스트 데이터를 사용해 **다음 토큰 예측**이라는 목표로 사전 학습(pre‑training)을 진행합니다. '
 '이후에는 인간이 만든 예시와 피드백을 활용해 **지도 학습(supervised fine‑tuning)** 및 **강화 '
 '학습(RLHF)**을 적용해 모델이 원하는 답변 형태와 품질을 갖도록 조정합니다. 이렇게 두 단계의 학습 과정을 거쳐, 다양한 언어 '
 '이해와 생성 능력을 갖춘 대규모 언어 모델이 완성됩니다.')
--------------------------------------------------
<class 'str'> Gemini 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.
('Gemini 모델은 대규모 텍스트와 멀티모달 데이터를 사용해 사전 학습(pre‑training) 단계에서 언어와 이미지 간의 관계를 '
 '학습합니다.  \n'
 '이후 인간 피드백을 활용한 강화 학습(RLHF) 과정을 거쳐, 질문에 대한 정확도와 안전성을 높이는 미세 조정(fine‑tuning)을 '
 '수행합니다.  \n'
 '학습 과정에서는 트랜스포머 아키텍처의 self‑attention 메커니즘을 이용해 문맥을 장기적으로 파악하고, 멀티모달 토큰을 통합해 '
 '의미를 종합합니다.  \n'
 '마지막으로, 지속적인 업데이트와 대규모 분산 학습 인프라를 활용해 모델의 성능을 지속적으로 개선합니다.')
--------------------------------------------------
<class 'str'> Claude 모델의 학습 원리를 5 문장으로 한국어로 답변해 주세요.
('Cla

##### 2) ChatPromptTemplate
* Tuple 형태의 system, user, assistant 메시지 지원
* 여러 개의 메시지를 조합하여 LLM에게 전달 가능
* 간결성과 가독성이 높고 단순한 구조

In [19]:
# 2-튜플 형태의 메시지 목록으로 프롬프트 생성 (type, content)

from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    # role, message
    ("system", "This system is an expert in answering questions about {topic}. \
     Please provide clear and detailed explanations."),
    ("user", "{model_name} 모델의 학습 원리를 설명해 주세요.이 AI 모델은 어느 나라에서 개발되었나요? 이 모델을 개발한 회사는 어디인가요?"),
])

messages = chat_prompt.format_messages(topic="AI", model_name="ExaOne")
print(messages)

# 생성한 메시지를 바로 주입하여 호출하기
llm = get_llm(temperature=0.7, provider="openai")
response = llm.invoke(messages)

print(type(response))
print(response.content)

[SystemMessage(content='This system is an expert in answering questions about AI.      Please provide clear and detailed explanations.', additional_kwargs={}, response_metadata={}), HumanMessage(content='ExaOne 모델의 학습 원리를 설명해 주세요.이 AI 모델은 어느 나라에서 개발되었나요? 이 모델을 개발한 회사는 어디인가요?', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.messages.ai.AIMessage'>
## EXAONE 모델이란?

**EXAONE(엑사원)**은 LG그룹의 인공지능 연구 조직인 **LG AI Research**가 개발한 생성형 AI·초거대 언어 모델 계열입니다. 문서 작성, 질의응답, 요약, 코드 생성, 이미지 이해 등 다양한 작업을 수행하도록 설계되었습니다. 버전에 따라 텍스트 중심 모델과 이미지·텍스트를 함께 처리하는 멀티모달 모델이 있습니다.

## 학습 원리

EXAONE의 기본적인 학습 방식은 GPT와 같은 **트랜스포머(Transformer) 기반 생성 모델**의 원리와 유사합니다.

### 1. 대규모 데이터로 사전 학습

먼저 책, 웹 문서, 논문, 기술 자료, 코드 등 대규모 데이터를 사용해 모델을 학습시킵니다.

문장 일부가 주어졌을 때 다음에 올 토큰(token)을 예측하는 방식입니다.

예를 들어,

> “인공지능은 데이터를 바탕으로 …”

라는 문장이 주어지면 모델은 다음에 올 가능성이 높은 단어 또는 토큰을 예측합니다.

이 과정을 매우 많은 문장에 대해 반복하면서 모델은 다음과 같은 패턴을 학습합니다.

- 단어와 문장의 의미
- 문법과 문맥
- 질문과 답변의 관계
- 전문 지식과 개념 간의 연관성
- 문서의 구조와 표현 방식

즉, 단순히 문장을 저장하는 것이 아니

In [20]:
# 체인을 생성하여 호출하기
chain = chat_prompt | llm | StrOutputParser()

response = chain.invoke({"topic":"AI", "model_name":"deepseek"})
print(type(response))
print(response)

<class 'langchain_core.messages.base.TextAccessor'>
## DeepSeek란 무엇인가요?

DeepSeek는 하나의 단일 모델 이름이라기보다, 중국의 AI 연구 기업이 개발한 **대규모 언어 모델(LLM) 계열**입니다. 대표적으로 다음과 같은 모델이 있습니다.

- **DeepSeek-V3**: 일반적인 대화·코딩·추론을 수행하는 대규모 언어 모델
- **DeepSeek-R1**: 수학, 논리, 코딩 문제에서 추론 과정을 강화한 모델
- **DeepSeek-Coder**: 프로그래밍에 특화된 모델
- **DeepSeek-V2 등**: 이전 세대 모델

모델마다 세부 학습 방법은 다르지만, 기본 원리는 다른 최신 언어 모델과 비슷하면서도 **Mixture-of-Experts(MoE)**, 효율적인 어텐션 구조, 강화학습 등을 적극적으로 활용한다는 특징이 있습니다.

---

## 1. 기본 학습 원리: 다음 토큰 예측

DeepSeek 같은 언어 모델은 기본적으로 **Transformer** 구조를 사용합니다.

학습 과정에서는 인터넷 문서, 책, 코드 등 대규모 텍스트를 입력하고, 문맥을 바탕으로 다음에 올 단어 또는 토큰을 예측하도록 훈련합니다.

예를 들어 다음 문장이 입력되면:

> 인공지능은 데이터를 통해 패턴을

모델은 다음 토큰으로 `학습한다`, `발견한다` 등의 가능성을 계산합니다.

수많은 문장에 대해 예측을 반복하면서 모델은 다음과 같은 능력을 학습합니다.

- 문법과 문장 구조
- 단어와 개념 간의 관계
- 사실 정보와 지식
- 프로그래밍 언어의 문법
- 문제 해결에 필요한 패턴
- 여러 언어 간 번역 관계

이 과정은 흔히 **사전학습(pretraining)**이라고 합니다.

---

## 2. Transformer와 어텐션

DeepSeek는 Transformer 계열 모델입니다. Transformer의 핵심 기술은 **어텐션(attention)**입니다.

어텐션은 문장 안에서 현재 단어를 이해

#### 3) ChatPromptTemplate
* SystemMessagePromptTemplate와 HumanMessagePromptTemplate 클래스 사용
* 객체 지향적 접근 - Message 객체를 독립적으로 생성 가능
* 여러 조건에 따라 다른 시스템 메시지 선택

```python
if user_is_beginner:
    system_message = SystemMessagePromptTemplate.from_template("초보자를 위한 설명: {topic}")
else:
    system_message = SystemMessagePromptTemplate.from_template("전문가를 위한 상세 분석: {topic}")
```

In [21]:
# ChatMessagePromptTemplate 활용

from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
    ChatMessagePromptTemplate
)
from langchain_openai import ChatOpenAI

# 개별 메시지 템플릿 정의
system_message = SystemMessagePromptTemplate.from_template(
    "You are an AI expert in {topic}. Please provide clear and detailed explanations."
)
user_message = HumanMessagePromptTemplate.from_template(
    "{question}"
)
ai_message = AIMessagePromptTemplate.from_template(
    "This is an example answer about {topic}."
)

# ChatPromptTemplate로 메시지들을 묶기
chat_prompt = ChatPromptTemplate.from_messages([
    system_message,
    user_message,
    ai_message
])

# 메시지 생성
messages = chat_prompt.format_messages(topic="건강", question="독감예방 접종을 꼭 맞아야 하나요?")

# LLM 호출 (공통 설정 셀의 llm 사용)
llm = get_llm(temperature=0.7, provider="openai")
response = llm.invoke(messages)

# 결과 출력
print(response.content)

독감(인플루엔자) 예방접종은 **모든 사람이 법적으로 꼭 맞아야 하는 것은 아니지만, 대부분의 사람에게 권장**됩니다. 독감은 단순 감기보다 고열·폐렴·입원 등 합병증을 일으킬 수 있고, 예방접종은 감염과 중증 위험을 줄이는 가장 효과적인 방법 중 하나입니다.

### 특히 꼭 권장되는 분
- **65세 이상**
- **생후 6개월~만 18세 어린이·청소년**
- **임신부**
- 천식·만성폐질환, 심장질환, 당뇨, 신장·간질환, 면역저하 등 만성질환자
- 요양시설 거주자
- 영유아·노인·면역저하자와 함께 생활하거나 이들을 돌보는 사람
- 의료·돌봄 종사자

### 접종 시기
- 보통 **독감 유행 전인 가을**, 대개 10~11월에 맞습니다.
- 유행이 시작된 뒤라도 접종하지 않았다면 맞는 것이 도움이 됩니다.
- 독감 바이러스가 매년 변하고 예방 효과가 시간이 지나며 감소하므로 **매년 접종**합니다.

### 안전성
접종 후 주사 부위 통증, 미열, 피로감이 1~2일 나타날 수 있지만 대부분 가볍습니다. **예방접종 때문에 독감에 걸리지는 않습니다.**

### 접종 전 의사와 상담해야 하는 경우
- 이전 독감 백신 접종 후 심한 알레르기 반응이 있었던 경우
- 과거 접종 후 6주 이내에 길랭-바레 증후군이 발생한 경우
- 현재 고열을 동반한 급성 질환이 있는 경우  
  *(가벼운 감기 증상만으로는 대개 접종을 미루지 않습니다.)*

따라서 특별한 금기사항이 없다면 **매년 맞는 것을 권합니다.** 특히 임신부, 어린이, 고령자, 만성질환자는 접종의 이득이 더 큽니다.


#### ChatMessagePromptTemplate는 여러 종류의 메시지(시스템, 인간, AI)를 조합하여 복잡한 프롬프트를 생성할 때 유용합니다.
* SystemMessagePromptTemplate: 이 템플릿은 AI 모델에게 역할을 부여하거나 전반적인 규칙을 설정하는 시스템 메시지를 만듭니다. 위의 예시에서는 "번역을 도와주는 유용한 도우미"라는 역할을 지정합니다.
* HumanMessagePromptTemplate: 이 템플릿은 사용자의 질문이나 요청을 담는 인간 메시지를 만듭니다. 아래의 예시에서는 번역할 텍스트를 입력받습니다.
* ChatPromptTemplate.from_messages: 이 클래스 메서드는 시스템 메시지, 인간 메시지 등 여러 종류의 MessagePromptTemplate 객체들을 리스트로 받아 하나의 채팅 프롬프트 템플릿으로 통합합니다.
* format_messages: 이 메서드는 정의된 템플릿에 실제 값을 채워 넣어 [SystemMessage, HumanMessage] 형태의 리스트를 반환합니다. 이 리스트는 채팅 모델(Chat Model) 에 바로 전달될 수 있습니다.

In [ ]:
# 필요한 라이브러리 임포트
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

# 1. SystemMessagePromptTemplate와 HumanMessagePromptTemplate 생성
# SystemMessagePromptTemplate는 모델의 페르소나 또는 기본 지침을 설정합니다.
system_template = "You are a helpful assistant that translates {input_language} to {output_language}."
system_message_prompt = SystemMessagePromptTemplate.from_template(system_template)

# HumanMessagePromptTemplate는 사용자로부터 받는 입력 프롬프트를 정의합니다.
human_template = "{text_to_translate}"
human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

# 2. ChatPromptTemplate 생성
# 위에서 만든 두 템플릿을 리스트로 묶어 ChatPromptTemplate을 만듭니다.
chat_prompt_template = ChatPromptTemplate.from_messages([system_message_prompt, human_message_prompt])

# 3. 프롬프트 포맷팅
# chat_prompt_template.format_messages()를 사용하여 최종 메시지 리스트를 생성합니다.
# 이 함수는 딕셔너리 형태의 입력 변수를 받습니다.
formatted_prompt = chat_prompt_template.format_messages(
    input_language="English",
    output_language="Korean",
    text_to_translate="Chaining can mean making multiple LLM calls in a sequence. Language models are often non deterministic and can make errors, so making multiple calls to check previous outputs or to break down larger tasks into bite-sized steps can improve results."
)

# 4. 결과 출력
print(formatted_prompt)

# LLM 호출
response = llm.invoke(formatted_prompt)

# 결과 출력
print(response.content)

##### 4) FewShotPromptTemplate
* FewShotPromptTemplate은 모델이 특정 형식을 따르게 하거나, 일관된 응답을 생성하도록 유도할 때 유용합니다.
* 도메인 지식이 필요하거나, AI가 오답을 줄이고 더 신뢰할 만한 답변을 생성하도록 해야 할 때 효과적입니다.

##### 4-1) PromptTemplate을 사용하지 않는 경우

In [22]:
# PromptTemplate을 사용하지 않는 경우

# model
print(llm.model_name)

# chain 실행
result = llm.invoke("태양계의 행성들을 간략히 정리해 주세요.")

print(type(result))
print(result.content)

gpt-5.6-luna
<class 'langchain_core.messages.ai.AIMessage'>
태양계의 행성은 태양에서 가까운 순서대로 **8개**입니다.

| 순서 | 행성 | 특징 |
|---|---|---|
| 1 | **수성** | 태양에 가장 가까운 작은 암석 행성. 대기가 희박하고 일교차가 큼 |
| 2 | **금성** | 두꺼운 이산화탄소 대기로 인해 태양계에서 가장 뜨거운 행성 |
| 3 | **지구** | 액체 상태의 물과 생명체가 존재하는 것으로 알려진 행성 |
| 4 | **화성** | ‘붉은 행성’. 과거 물이 흘렀던 흔적이 있으며 탐사가 활발함 |
| 5 | **목성** | 가장 큰 행성. 거대한 가스 행성이며 대적점이라는 폭풍이 있음 |
| 6 | **토성** | 아름답고 넓은 고리로 유명한 가스 행성 |
| 7 | **천왕성** | 자전축이 크게 기울어 옆으로 누운 듯 회전하는 얼음 거대 행성 |
| 8 | **해왕성** | 태양에서 가장 먼 행성. 강한 바람이 부는 푸른 얼음 거대 행성 |

- **지구형 행성**: 수성, 금성, 지구, 화성  
  → 작고 단단한 암석으로 이루어짐
- **목성형 행성**: 목성, 토성, 천왕성, 해왕성  
  → 크고 두꺼운 대기를 지님
- **명왕성**은 2006년부터 행성이 아닌 **왜행성**으로 분류됩니다.


##### 4-2) FewShotChatMessagePromptTemplate 사용하는 경우

In [23]:
# FewShotChatMessagePromptTemplate 사용하는 경우
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# 예시는 시스템 프롬프트(초등학생 눈높이)와 같은 톤·같은 형식(제목 + 비유 글머리표 + 한 줄 정리)으로 통일
examples_list = [
    {
        "input": "뉴턴의 운동 법칙을 요약해 주세요.",
        "output": """### 뉴턴의 운동 법칙
- **관성**: 달리던 버스가 갑자기 멈추면 몸이 앞으로 쏠리죠? 움직이던 건 계속 움직이려 해요.
- **가속도**: 같은 힘으로 밀면 가벼운 공이 무거운 공보다 더 빨리 굴러가요.
- **작용-반작용**: 벽을 밀면 벽도 나를 똑같이 밀어요.
 한 줄 정리: 힘은 항상 짝으로 움직여요!"""
    },
    {
        "input": "지구의 대기 구성 요소를 알려주세요.",
        "output": """### 지구 대기의 구성
- **질소 (78%)**: 공기의 대부분이에요. 거의 반응하지 않는 얌전한 기체예요.
- **산소 (21%)**: 우리가 숨 쉴 때 꼭 필요해요.
- **이산화탄소 (0.04%)**: 아주 조금이지만 식물이 자라는 데 필요해요.
 한 줄 정리: 공기의 거의 전부는 질소와 산소예요!"""
    }
]

# 예제 프롬프트 템플릿
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

# FewShotChatMessagePromptTemplate 적용
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples_list,
)

# 최종 프롬프트 구성
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 초등학생도 쉽게 이해할 수 있도록 쉽게 설명하는 과학 교육자입니다."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

# 모델 생성 및 체인 구성
chain = final_prompt | llm
print(llm.model_name)
print(chain)

# 테스트 실행
result = chain.invoke({"input": "전기의 발생 원리에 대하여 설명해 줘"})
print(result.content)

gpt-5.6-luna
first=ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 초등학생도 쉽게 이해할 수 있도록 쉽게 설명하는 과학 교육자입니다.'), additional_kwargs={}), FewShotChatMessagePromptTemplate(examples=[{'input': '뉴턴의 운동 법칙을 요약해 주세요.', 'output': '### 뉴턴의 운동 법칙\n- **관성**: 달리던 버스가 갑자기 멈추면 몸이 앞으로 쏠리죠? 움직이던 건 계속 움직이려 해요.\n- **가속도**: 같은 힘으로 밀면 가벼운 공이 무거운 공보다 더 빨리 굴러가요.\n- **작용-반작용**: 벽을 밀면 벽도 나를 똑같이 밀어요.\n 한 줄 정리: 힘은 항상 짝으로 움직여요!'}, {'input': '지구의 대기 구성 요소를 알려주세요.', 'output': '### 지구 대기의 구성\n- **질소 (78%)**: 공기의 대부분이에요. 거의 반응하지 않는 얌전한 기체예요.\n- **산소 (21%)**: 우리가 숨 쉴 때 꼭 필요해요.\n- **이산화탄소 (0.04%)**: 아주 조금이지만 식물이 자라는 데 필요해요.\n 한 줄 정리: 공기의 거의 전부는 질소와 산소예요!'}], input_variables=[], input_types={}, partial_variables={}, example_prompt=ChatPromptTemplate(input_variables=['input', 'output'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate

#### 5-1) PartialPrompt 
* 프롬프트를 더 동적으로 활용할 수 있으며, AI 응답을 더 일관성 있게 조정 가능함

In [24]:
from datetime import datetime
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 계절을 결정하는 함수 (남반구/북반구 고려)
def get_current_season(hemisphere="north"):
    month = datetime.now().month

    if hemisphere == "north":  # 북반구 (기본값)
        if 3 <= month <= 5:
            return "봄"
        elif 6 <= month <= 8:
            return "여름"
        elif 9 <= month <= 11:
            return "가을"
        else:
            return "겨울"
    else:  # 남반구 (계절 반대)
        if 3 <= month <= 5:
            return "가을"
        elif 6 <= month <= 8:
            return "겨울"
        elif 9 <= month <= 11:
            return "봄"
        else:
            return "여름"

# 프롬프트 템플릿 정의 (부분 변수 적용)
prompt = PromptTemplate(
    template="{season}에 일어나는 대표적인 지구과학 현상은 {phenomenon}이 맞나요? \
        {season}에 주로 발생하는 지구과학 현상을 3개 알려주세요",
    input_variables=["phenomenon"],  # 사용자 입력 필요
    # 함수 자체를 전달 → format() 호출 시점마다 현재 계절을 계산
    partial_variables={"season": get_current_season}  # 동적으로 계절 값 할당
)


# 특정 계절의 현상 질의
query = prompt.format(phenomenon="태풍 발생")
llm = get_llm(temperature=0.7, provider="openai")  # 제공자/키/모델명은 공통 설정 셀의 get_llm 사용
print(llm.model_name)
result = llm.invoke(query)


# 결과 출력
print(f" 프롬프트: {query}")
print(f" 모델 응답: {result.content}")

gpt-5.6-luna
 프롬프트: 가을에 일어나는 대표적인 지구과학 현상은 태풍 발생이 맞나요?         가을에 주로 발생하는 지구과학 현상을 3개 알려주세요
 모델 응답: 네. **태풍은 가을에도 대표적으로 발생하지만, 주로 늦여름부터 초가을(8~10월)**에 많이 나타납니다. 다만 태풍은 여름에만 발생하는 현상은 아닙니다.

가을에 주로 볼 수 있는 지구과학 현상 3가지는 다음과 같습니다.

1. **태풍 발생 및 북상**
   - 여름 동안 데워진 바다에서 발생한 태풍이 가을 초까지 우리나라에 영향을 줍니다.
   - 우리나라에는 주로 8~9월에 태풍이 많이 옵니다.

2. **추분**
   - 매년 9월 22~23일경 발생하며, 태양이 적도 부근을 비추어 **낮과 밤의 길이가 거의 같아지는 현상**입니다.
   - 추분 이후에는 북반구에서 밤이 점점 길어집니다.

3. **서리와 첫얼음**
   - 가을이 깊어지면서 밤 기온이 크게 내려가 지표면의 수증기가 얼어 **서리**가 생깁니다.
   - 기온이 더 낮은 지역이나 산간에서는 **첫얼음**도 나타납니다.

즉, 태풍은 가을의 대표적인 현상 중 하나이지만, **추분과 서리·첫얼음**도 가을을 대표하는 중요한 지구과학 현상입니다.


In [25]:
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 계절을 결정하는 함수 (남반구/북반구 고려)
def get_current_season(hemisphere="north"):
    month = datetime.now().month

    if hemisphere == "north":  # 북반구 (기본값)
        if 3 <= month <= 5:
            return "봄"
        elif 6 <= month <= 8:
            return "여름"
        elif 9 <= month <= 11:
            return "가을"
        else:
            return "겨울"
    else:  # 남반구 (계절 반대)
        if 3 <= month <= 5:
            return "가을"
        elif 6 <= month <= 8:
            return "겨울"
        elif 9 <= month <= 11:
            return "봄"
        else:
            return "여름"

# Step 1: 현재 계절 결정
#season_name = get_current_season()
season_name = get_current_season("south")  # 계절 값 얻기
print(f"현재 계절: {season_name}")

현재 계절: 봄


In [26]:
# Step 2: 해당 계절의 자연 현상 추천
prompt2 = ChatPromptTemplate.from_template(
    "{season}에 주로 발생하는 대표적인 지구과학 현상 3가지를 알려주세요. "
    "각 현상에 대해 간단한 설명을 포함해주세요."
)

print(llm.model_name)
# 체인 2: 자연 현상 추천 (입력: 계절 → 출력: 자연 현상 목록)
chain2 = (
    {"season": lambda x : season_name}  # chain2의 출력을 season 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

# 실행: 현재 계절에 따른 자연 현상 추천
response = chain2.invoke({})
print(f"\n {season_name}에 발생하는 자연 현상:\n{response}")

gpt-5.6-luna

 봄에 발생하는 자연 현상:
봄에 주로 나타나는 대표적인 지구과학 현상은 다음과 같습니다.

1. **황사**
   - 봄철 중국과 몽골의 건조한 사막 지역에서 발생한 모래먼지가 강한 바람을 타고 우리나라까지 이동하는 현상입니다.
   - 시야를 흐리게 하고 호흡기 건강에 영향을 줄 수 있습니다.

2. **꽃샘추위**
   - 겨울이 끝난 뒤에도 일시적으로 차가운 대륙성 고기압이 남하하면서 기온이 다시 크게 떨어지는 현상입니다.
   - 주로 3~4월에 나타나며, 개화한 식물이나 농작물에 피해를 줄 수 있습니다.

3. **춘분**
   - 태양이 적도 상공을 지나면서 낮과 밤의 길이가 거의 같아지는 천문 현상입니다.
   - 보통 3월 20일 또는 21일경에 발생하며, 춘분 이후에는 점차 낮의 길이가 밤보다 길어집니다.


#### 5-2) PartialPrompt 
* API 호출 데이터, 시간 정보, 사용자 정보 등을 반영할 때 매우 유용함

In [27]:
import requests
from datetime import datetime
from langchain_core.prompts import PromptTemplate

EXCHANGE_API_URL = "https://api.exchangerate-api.com/v4/latest/USD"


def get_exchange_rate() -> str:
    """USD→KRW 환율과 API 기준 일자를 문자열로 반환합니다."""
    response = requests.get(EXCHANGE_API_URL, timeout=10)
    response.raise_for_status()
    data = response.json()
    base_date = data.get("date", datetime.now().strftime("%Y-%m-%d"))
    return f"{base_date} 기준 1달러 = {data['rates']['KRW']}원"


# Partial Prompt 활용 (함수 자체를 전달 → format() 호출 시점마다 실행)
prompt = PromptTemplate(
    template=(
        "오늘 날짜: {today}\n"
        "환율 정보: {info}\n\n"
        "위 데이터만 근거로 환율 상황을 간단히 분석해 주세요. "
        "제공되지 않은 과거 수치나 뉴스는 추측해서 쓰지 마세요."
    ),
    input_variables=[],  # 사용자 입력 없음
    partial_variables={
        "today": lambda: datetime.now().strftime("%Y-%m-%d"),
        "info": get_exchange_rate,  # API에서 가져온 데이터 자동 반영
    },
)

# LLM 모델명
print(llm.model_name)

# 프롬프트를 한 번만 생성해 재사용 (API 중복 호출 방지)
formatted = prompt.format()

# 모델에 프롬프트 전달 및 응답 받기
response = llm.invoke(formatted)

# 결과 출력
print(" 프롬프트:", formatted)
print(" 모델 응답:", response.content)

gpt-5.6-luna
 프롬프트: 오늘 날짜: 2026-10-08
환율 정보: 2026-10-08 기준 1달러 = 1339.09원

위 데이터만 근거로 환율 상황을 간단히 분석해 주세요. 제공되지 않은 과거 수치나 뉴스는 추측해서 쓰지 마세요.
 모델 응답: 2026년 10월 8일 기준 환율은 **1달러 = 1,339.09원**입니다.

- 달러를 원화로 환전하면: **100달러 = 133,909원**
- 원화를 달러로 환전하면: **100만 원 ≈ 746.78달러**
- 비교 가능한 과거 환율이나 추가 정보가 없으므로, 환율의 상승·하락 추세나 고점·저점 여부는 판단할 수 없습니다.
